# MTP — Multi-Token Prediction, explained from scratch

You asked: *"when the main model runs with input `[T, G1, G2, G3]`, it just produces P4 (the next token after G3). How does it also verify G1, G2, G3?"*

The single most important fact, which almost nobody states clearly:

> **A transformer forward pass produces a separate prediction at every input position, not just the last one.**

When you feed in 4 tokens, you get back 4 predictions — one for "what comes after position 0", one for "what comes after position 1", etc. They are all computed **in parallel** in the same matrix multiplications. During training, transformers learn from the loss at every position (this is called *teacher forcing*). At inference time, we usually throw away all but the last position. **Speculative decoding uses those normally-discarded predictions to verify guesses.**

Let's prove it with a tiny runnable model.

## A toy "transformer"

Real transformers are huge tensor operations. Ours is a hand-programmed lookup that captures the **essential property**: input N tokens → output N predictions.

In [ ]:
KNOWN_SEQUENCES = [
    "the cat sat on the mat .".split(),
    "the dog ran in the park .".split(),
    "the sun rose in the east .".split(),
]

def toy_transformer(input_tokens):
    """Return the most likely next token at EACH input position.
    
    Returns a list of predictions, one per input position. The prediction
    at index i is what the model thinks comes AFTER position i.
    """
    predictions = []
    for i in range(len(input_tokens)):
        prefix = input_tokens[:i+1]
        next_token = None
        for known in KNOWN_SEQUENCES:
            if known[:len(prefix)] == prefix and len(known) > len(prefix):
                next_token = known[len(prefix)]
                break
        predictions.append(next_token)
    return predictions

# Feed in 4 tokens; we get 4 predictions back.
tokens = ["the", "cat", "sat", "on"]
predictions = toy_transformer(tokens)

print("Position | Input    | What the model predicts next")
print("-" * 50)
for i, (tok, pred) in enumerate(zip(tokens, predictions)):
    print(f"  {i}      | {tok!r:<8} | {pred!r}")


**See it?** 4 inputs → 4 predictions. In normal autoregressive decoding, we'd only look at the last row's prediction (`'the'`) and ignore the other three.

That's the waste speculative decoding fixes.

## Vanilla autoregressive decoding (the slow baseline)

Here's how plain decoding works — **one main-model pass per generated token**:

In [ ]:
def vanilla_decode(seed_tokens, num_to_generate):
    """Standard autoregressive: one main-model pass per token."""
    tokens = list(seed_tokens)
    main_passes = 0
    for _ in range(num_to_generate):
        predictions = toy_transformer(tokens)  # one expensive main-model pass
        main_passes += 1
        next_token = predictions[-1]           # only the LAST position is used
        if next_token is None:
            break
        tokens.append(next_token)
    return tokens, main_passes

tokens, passes = vanilla_decode(["the", "cat"], 5)
print(f"Generated: {' '.join(t for t in tokens if t is not None)}")
print(f"Main-model passes: {passes}")
print(f"Tokens per pass: {(len(tokens) - 2) / passes:.2f}")


5 tokens → 5 main-model passes. Each pass loads all the model's weights from HBM memory; that load is the actual bottleneck on a GPU/NPU, not the math. So one token per HBM load is *slow*.

## The speculative idea: guess + verify in one pass

What if a *cheap* helper could guess the next 3 tokens, and then we use the main model to verify all of them in a **single** main-model pass?

- All 3 guesses correct → we get 4 tokens (3 guesses + 1 free bonus) from one pass.
- 2 guesses correct → 3 tokens.
- 1 guess correct → 2 tokens.
- 0 guesses correct → 1 token (the model's correction).

We *never* do worse than vanilla. We sometimes do much better.

## Introducing MTP heads — the cheap helpers

In MTP, the "helper" isn't a separate model. It's a few extra tiny modules bolted onto the main model itself, trained jointly. We'll fake them with a small lookup that's mostly right but sometimes wrong.

In [ ]:
# The MTP heads: cheap, fast, fallible.
GOOD_GUESSES = {
    ("the", "cat"): ["sat", "on", "the"],    # all 3 will turn out right
    ("the", "dog"): ["barked", "loudly", "."], # first one is WRONG
    ("the", "sun"): ["rose", "in", "the"],
}

def mtp_heads(recent_context):
    """3 guesses for the next 3 tokens. Cheap but fallible."""
    key = tuple(recent_context[-2:])
    return GOOD_GUESSES.get(key, [None, None, None])

print("After 'the cat', MTP guesses:", mtp_heads(["the", "cat"]))
print("After 'the dog', MTP guesses:", mtp_heads(["the", "dog"]))


## ★ The key step — one main-model pass verifies all 3 guesses

**This is the part you couldn't picture.** Let's walk through it explicitly.

State:
- Accepted prefix so far: `["the", "cat"]`
- MTP guesses: `G1="sat"`, `G2="on"`, `G3="the"`

We construct the main-model input as **the last accepted token + the 3 guesses**:

```
input = ["the", "cat", "sat", "on", "the"]
         └─prefix─┘   └────── guesses ──────┘
```

We run the main model **once** on this 5-token input. It returns 5 predictions — one per position. The interesting positions are 1, 2, 3, 4:

In [ ]:
accepted_prefix = ["the", "cat"]
guesses = ["sat", "on", "the"]
main_input = accepted_prefix + guesses

# One main-model pass.
predictions = toy_transformer(main_input)

print("Main model input + per-position output:")
print()
print("  Position | Input    | Model says comes next | Role")
print("  " + "-" * 65)
roles = ["(throwaway)", "P1: verifies G1", "P2: verifies G2", "P3: verifies G3", "P4: BONUS"]
for i, (tok, pred, role) in enumerate(zip(main_input, predictions, roles)):
    print(f"    {i}    | {tok!r:<8} | {pred!r:<22} | {role}")

print()
print("Verification:")
p1, p2, p3, p4 = predictions[1], predictions[2], predictions[3], predictions[4]
g1, g2, g3 = guesses
print(f"  G1={g1!r} vs P1={p1!r}: {'✓ ACCEPT' if g1 == p1 else '✗ REJECT'}")
print(f"  G2={g2!r}  vs P2={p2!r}:  {'✓ ACCEPT' if g2 == p2 else '✗ REJECT'}")
print(f"  G3={g3!r} vs P3={p3!r}: {'✓ ACCEPT' if g3 == p3 else '✗ REJECT'}")
print(f"  P4={p4!r} is the BONUS — no guess to compare, always free.")
print()
print(f"Result: {g1!r} {g2!r} {g3!r} {p4!r}  → 4 tokens from ONE main-model pass.")


**Now you can see it.** The transformer computed predictions for *every* input position in one forward pass. We compare:

- **P1** (model's prediction for what comes after `"cat"`) vs **G1** (what the MTP head guessed). If they match, G1 was right.
- **P2** (model's prediction for what comes after `"cat", "sat"`) vs **G2**. Only meaningful if we accepted G1 — but it was computed for free anyway.
- **P3** similar.
- **P4** is the **bonus**: the model's prediction for what comes after all the guesses. If all guesses are accepted, P4 is the actual next token to add — and we got it for free.

**This is why a main-model pass can emit up to `num_speculative_tokens + 1` tokens.** With `num_speculative_tokens=3` (your GLM-5.1 deployment), that's up to **4 tokens per pass**.

## What happens when a guess is wrong?

In [ ]:
accepted_prefix = ["the", "dog"]
guesses = mtp_heads(accepted_prefix)  # ['barked', 'loudly', '.'] — WRONG
main_input = accepted_prefix + guesses

predictions = toy_transformer(main_input)

print(f"Accepted prefix: {accepted_prefix}")
print(f"MTP guesses:     {guesses}")
print(f"Main input:      {main_input}")
print()
print("  Position | Input    | Model says comes next")
print("  " + "-" * 45)
for i, (tok, pred) in enumerate(zip(main_input, predictions)):
    print(f"    {i}    | {tok!r:<8} | {pred!r}")

print()
p1 = predictions[1]
g1 = guesses[0]
print(f"  G1={g1!r} vs P1={p1!r}: {'✓ ACCEPT' if g1 == p1 else '✗ REJECT (mismatch at the very first guess)'}")
print()
print(f"Because G1 was rejected, we use the model's actual prediction P1={p1!r}.")
print(f"G2 and G3 are DISCARDED — they were conditioned on G1 being right.")
print(f"P2, P3 are also discarded for the same reason.")
print()
print(f"Result: just {p1!r}  → 1 token from this main-model pass. Same as vanilla.")


So the worst case is **1 token per pass**, same as vanilla decoding. MTP can only make things faster, never slower (in token count — there's a small overhead from running the MTP heads, but they're tiny).

## How big is the speedup?

Let **α** = the *acceptance rate* per guess (probability that a single MTP guess matches the main model).

With `num_speculative_tokens = k = 3`:

- All 3 accepted (prob α³): 4 tokens
- First 2 accepted, 3rd rejected (prob α² · (1−α)): 3 tokens  
- First accepted, 2nd rejected (prob α · (1−α)): 2 tokens
- First rejected (prob 1−α): 1 token

Expected tokens per pass = **1 + α + α² + α³**.

In [ ]:
def expected_tokens_per_pass(alpha, k=3):
    """Expected tokens generated per main-model pass with acceptance rate alpha."""
    return sum(alpha**i for i in range(k + 1))

print(f"  α (acceptance) | Tokens/pass | Speedup vs vanilla")
print(f"  " + "-" * 50)
for alpha in [0.0, 0.3, 0.5, 0.7, 0.85, 0.95, 1.0]:
    eff = expected_tokens_per_pass(alpha)
    print(f"      {alpha:.2f}       |    {eff:.2f}     |     {eff:.1f}×")


**Realistic acceptance rates for production deployments are 80-95%**, so you typically see **3-3.7× faster decode** from MTP.

## Back to your benchmark

When you ran `yabench glm1` against the GLM-5.1 deployment and saw:

- **Output TPS ≈ 200 tokens/sec at c=8**
- **"ITL" ≈ 50ms per chunk**

What was actually happening:

1. The vLLM server does ~50-70 main-model passes per second per active sequence.
2. Each pass emits **1-4 tokens** (typically 3 on average, given GLM's acceptance rate).
3. The server emits **one SSE chunk per main-model pass** — so each chunk carries 1-4 tokens.
4. yabench's "ITL" measures the gap between SSE chunks → it's really *inter-pass* latency, not per-token latency.
5. Real per-token decode is ~**15-20ms**, not 50ms.

**If you turned MTP off** (relaunched the server without `--speculative-config`), your decode TPS would roughly drop by 3× — from 200 to ~70 tokens/sec at c=8. That's how much of your observed throughput is attributable to MTP.

This also explains why **the model's output quality is unchanged with MTP on**: rejected guesses are *thrown away*, not used. MTP is exact, not approximate.

## MTP vs other speculative-decoding schemes

| Method | Helper | Acceptance | Portability |
|---|---|---|---|
| Classical spec-dec | Separate small "draft" model | Medium | Portable but needs a paired small model |
| Medusa | Multiple parallel heads predict independently | Lower (heads don't condition on each other) | Bolt-on after training |
| EAGLE | Cascaded heads using hidden states | High | Bolt-on after training |
| **MTP** | Cascaded heads trained jointly with the main model | **Highest** for that specific model | Model-specific, baked in |

**Short version of everything above:** Speculative decoding is "guess ahead, verify in bulk". MTP lets the model do the guessing itself, using small heads that ride along on the main forward pass. The trick that makes verification work is that a transformer always emits a prediction at every input position in parallel — speculative decoding just stops throwing those predictions away.